# Sprint 01 — Acquisition des données (raw → bronze)

Première étape du médaillon : charger les **sources brutes** fournies dans la couche
`bronze` (fidèle à la source). C'est le **seul** endroit où le brut est lu ; les
analyses (`01_analyse_exploratoire.ipynb`) lisent ensuite **bronze**.

Sources brutes (`data/raw/`) :
- `telemetry.csv` — relevés capteurs horaires ;
- `releves_incidents.csv` — incidents (données personnelles, anonymisées au stade analyse) ;
- `machine.sql` — référentiel machines & maintenance (dump SQL).

> **Prérequis** : base PostgreSQL démarrée (conteneur `docker/pgdocker`).

## 1. Aperçu des sources brutes

Coup d'œil direct aux CSV bruts (avant chargement) : dimensions, types, premières lignes.

In [1]:
from indusense.data import loaders

telemetry = loaders.load_telemetry()
incidents = loaders.load_incidents()
telemetry.shape, incidents.shape

((135626, 7), (1245, 18))

In [2]:
telemetry.head()

,machine_id,timestamp,temperature_c,pressure_bar,voltage_mean_v,rotation_mean_rpm,pieces_produced
0,MACH-01,2025-06-01 00:00:00,46.348,198.203,227.568,1541.787,4
1,MACH-01,2025-06-01 00:00:00,46.332,198.206,227.570,1541.760,4
2,MACH-01,2025-06-01 01:00:00,48.762,198.295,227.480,1537.860,4
3,MACH-01,2025-06-01 02:00:00,51.352,199.545,228.680,1584.660,13
4,MACH-01,2025-06-01 03:00:00,49.512,201.641,228.440,1588.960,10


In [3]:
telemetry.info()
telemetry.isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 135626 entries, 0 to 135625
Data columns (total 7 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   machine_id         135626 non-null  str    
 1   timestamp          135626 non-null  str    
 2   temperature_c      134732 non-null  float64
 3   pressure_bar       134631 non-null  float64
 4   voltage_mean_v     135626 non-null  float64
 5   rotation_mean_rpm  134668 non-null  float64
 6   pieces_produced    135626 non-null  int64  
dtypes: float64(4), int64(1), str(2)
memory usage: 10.6 MB


machine_id             0
timestamp              0
temperature_c        894
pressure_bar         995
voltage_mean_v         0
rotation_mean_rpm    958
pieces_produced        0
dtype: int64

## 2. Chargement raw → bronze

Le **schéma** (tables typées bronze/silver) est créé par **Alembic** (`upgrade head`,
migrations/rollback — US 1.2). Puis on charge les données : référentiel (`machine.sql`)
+ CSV (idempotent, tables tronquées avant append). Équivalent CLI :
`uv run alembic upgrade head` puis `uv run indusense-ingest --migrate`.


In [4]:
# Schéma géré par Alembic (migrations/rollback, US 1.2) -> créer les tables typées.
from alembic import command
from alembic.config import Config

from indusense import config as cfg
from indusense.data import ingest
from indusense.data.db import get_engine

alembic_cfg = Config(str(cfg.PROJECT_ROOT / 'alembic.ini'))
alembic_cfg.set_main_option('script_location', str(cfg.PROJECT_ROOT / 'alembic'))
command.upgrade(alembic_cfg, 'head')
print('Schéma migré (alembic upgrade head)')

engine = get_engine()
ingest.setup_bronze(engine, migrate=True)  # données de référence (machine.sql)
for name in ('telemetry', 'incidents'):
    n = ingest.load_csv_to_bronze(engine, ingest.SOURCES[name])
    print(f'{name}: {n} lignes -> bronze.{ingest.SOURCES[name].table}')


INFO  [alembic.runtime.migration] Context impl PostgresqlImpl.


INFO  [alembic.runtime.migration] Will assume transactional DDL.


Schéma migré (alembic upgrade head)
Données de référence (machine, maintenance) chargées dans bronze.


telemetry: 135626 lignes -> bronze.telemetry
incidents: 1245 lignes -> bronze.incident


## 3. Contrôle de chargement (réconciliation)

Vérifie que `bronze` reflète fidèlement le brut (comptes raw vs bronze).

In [5]:
from indusense.data import loaders
from indusense.data.db import read_bronze

print("Réconciliation raw vs bronze :")
for label, raw_df, table in (
    ("telemetry", loaders.load_telemetry(), "telemetry"),
    ("incidents", loaders.load_incidents(), "incident"),
):
    n_raw, n_bronze = len(raw_df), len(read_bronze(table))
    print(f"  {label:10s} raw={n_raw:>6} bronze={n_bronze:>6}  "
          f"{'OK' if n_raw == n_bronze else 'ECART'}")

print("Référentiel (chargé depuis machine.sql) :")
for table in ("machine", "maintenance"):
    print(f"  bronze.{table}: {len(read_bronze(table))} lignes")

Réconciliation raw vs bronze :


  telemetry  raw=135626 bronze=135626  OK
  incidents  raw=  1245 bronze=  1245  OK
Référentiel (chargé depuis machine.sql) :
  bronze.machine: 15 lignes
  bronze.maintenance: 1562 lignes


## 4. Contrôle de structure (schéma)

Garde-fou à l'arrivée d'une nouvelle version des données : on compare la **structure**
des sources brutes (colonnes / types) à un **contrat versionné**
(`reports/schema_reference.json`, outil `indusense-schema`). Toute dérive (colonne
ajoutée/supprimée, type modifié) est signalée. Indépendant de la base (raw + JSON).


In [6]:
from indusense.data import schema

result = schema.check_drift()
print('Structure des sources brutes :')
for name, prof in result['current'].items():
    extra = f", {prof['n_rows']} lignes" if prof.get('n_rows') is not None else ' (DDL)'
    print(f'  {name}: {len(prof["columns"])} colonnes{extra}')
print()
print('Dérive vs contrat (reports/schema_reference.json) :')
for name, d in result['drift'].items():
    statut = 'conforme' if d.get('identique') else 'DERIVE -> ' + str(d)
    print(f'  {name}: {statut}')


Structure des sources brutes :
  telemetry: 7 colonnes, 135626 lignes
  incidents: 18 colonnes, 1245 lignes
  machine: 11 colonnes (DDL)
  maintenance: 11 colonnes (DDL)

Dérive vs contrat (reports/schema_reference.json) :
  telemetry: conforme
  incidents: conforme
  machine: conforme
  maintenance: conforme
